This notebook is mostly a sanity check that the isochrone works reasonably. 

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
import numpy as np

In [ ]:
from yasone.read_iso import ISOCMD

In [ ]:
from astropy.table import Table

In [ ]:
import matplotlib.pyplot as plt

In [ ]:

isos_fe_h = ["m1.00", "m1.25", "m1.50", "m1.75", "m2.00", "m2.50", "m3.00"]

isonames = [f"../MIST/MIST_v1.2_vvcrit0.4_SDSSugriz/MIST_v1.2_feh_{iso_fe_h}_afe_p0.0_vvcrit0.4_SDSSugriz.iso.cmd" for iso_fe_h in isos_fe_h]
isochrones = {fe_h: ISOCMD(name) for fe_h, name in zip(isos_fe_h, isonames)}



# Setup

The below sql creates the sample


```
SELECT u,g,r,i,z,ra,dec, flags_r, err_g, err_r, err_i, err_z
FROM Star
WHERE
ra BETWEEN 211.36371-0.5 and 211.36371+0.5 AND dec BETWEEN 28.53444-0.5 and 28.53444+0.5
AND ((flags_r & 0x10000000) != 0)
-- detected in BINNED1
AND ((flags_r & 0x8100000c00a4) = 0)
-- not EDGE, NOPROFILE, PEAKCENTER, NOTCHECKED, PSF_FLUX_INTERP,
-- SATURATED, or BAD_COUNTS_ERROR
AND (((flags_r & 0x400000000000) = 0) or (psfmagerr_r <= 0.2))
-- not DEBLEND_NOPEAK or small PSF error
-- (substitute psfmagerr in other band as appropriate)
AND (((flags_r & 0x100000000000) = 0) or (flags_r & 0x1000) = 0)
-- not INTERP_CENTER or not COSMIC_RAY
``` 

In [ ]:
ra0 = 211.36371
dec0 = 28.53444
coord0 = SkyCoord(ra0, dec0, unit="deg")

In [ ]:
df_all = Table.read("ngc_5466.csv")
r2 = (df_all["ra"] - ra0)**2 + (df_all["dec"] - dec0)**2


In [ ]:
df = df_all[r2 < 0.15**2]

In [ ]:
plt.hist(r2)

In [ ]:
plt.hist(df["g"], np.linspace(0, 28, 100))

In [ ]:
from yasone.selection import plot_iso
from yasone.analysis import get_extinction, fit_err
import yasone.selection as filter_utils

In [ ]:
filter_utils.load_isochrones()

In [ ]:
iso = isochrones["m1.50"][10.1]
iso = iso[iso["phase"] < 4]

In [ ]:
dm = 16.02
A = 0.05

In [ ]:
from dustmaps.sfd import SFDQuery

In [ ]:
params = filter_utils.filter_params(E_BV =Ebv, dm=dm, 
                                    iso_fe_h="m2.00",
                                    iso_log_age=10.2,
                                    iso_width=0.05
                                   )

In [ ]:
gr_err = fit_err(df["g"], df["err_g"] + df["err_r"])
ri_err = fit_err(df["r"], df["err_r"] + df["err_i"])

In [ ]:
julen_gr = [0.5718837290502794, 0.5575971834264433, 0.5417248603351956, 0.5359054934823091, 0.523510242085661, 0.4996944832402235, 0.48748836126629413, 0.45110277001862187, 0.4123312383612663, 0.3901885474860336, 0.39689536778398504, 0.4227915502793296, 0.46089385474860345, 0.49407879422718803, 0.5347416201117319, 0.5934444832402235, 0.6539658985102421, 0.74192562849162, 0.7974132914338921, 0.8390072160148976, 0.8810230446927374]
julen_g = [17.05372015344448, 17.478405460989805, 17.920944004434137, 18.358581659592467, 18.752286351901283, 19.338059189894835, 19.508073338292565, 19.70533409180414, 19.947344622879562, 20.29571609854286, 20.862177102933238, 21.446257967589958, 21.989031345264664, 22.447556119546668, 22.891903324144167, 23.44547032482971, 24.028617686955762, 24.779620472877376, 25.477530302367303, 26.216630931022916, 27.263028923992472]

julen_ri  = [0.3065634218289086, 0.3065634218289086, 0.29781710914454274, 0.2859734513274337, 0.2741740412979352, 0.24728613569321545, 0.20769911504424776, 0.20448377581120947, 0.21588495575221245, 0.25946902654867254, 0.3681858407079648, 0.4320501474926255, 0.47762536873156347]
julen_r = [16.702331577590336, 16.702331577590336, 17.34541972080003, 17.943894802755167, 18.841224354488194, 19.227595859884342, 19.84274927253306, 20.33361081439464, 20.785811632104313, 21.896482375041437, 23.561191940771298, 24.774290029098676, 26.585214925043278]

In [ ]:
Ebv * 3.1

In [ ]:
params

In [ ]:
Ebv = SFDQuery()(coord0)
A_g, A_r, A_i = get_extinction(Ebv)

In [ ]:
dm_ngc5466 = 16.02
dm_yasone1 = 15.74
E_BV_y1 = 1.0 / 3.1
A_g_y1, A_r_y1, A_i_y1 = get_extinction(E_BV_y1)

d_gr  = A_g_y1 - A_r_y1 - (A_g - A_r)
d_g = A_g_y1 - A_g + dm_yasone1 - dm_ngc5466


d_ri  = A_r_y1 - A_i_y1 - (A_r - A_i)
d_r = A_r_y1 - A_r  + dm_yasone1 - dm_ngc5466

In [ ]:
params.E_BV = E_BV_y1
params.dm = dm_yasone1
params.iso_log_age = 10.1
params.iso_fe_h = "m1.50"

In [ ]:
plt.scatter(df["g"] - df["r"] + d_gr, df["g"] + d_g, s=0.5, lw=0, zorder=-5)
plt.ylim(27, 16)
plt.xlim(-1, 2)
# plt.plot(iso["SDSS_g"] - iso["SDSS_r"] + A_g - A_r, iso["SDSS_g"] + A_g + dm, color="k")
filter_utils.plot_iso_gr(params,)


plt.plot(julen_gr, julen_g, label="paper isochrone", color="C1")

plt.plot([], [], label="g-r isochrone", color="grey")

plt.gca().set_box_aspect(1)
plt.xlabel("$g - r$ (mag)")
plt.ylabel("$g$ (mag)")
plt.title("NGC 5466 (shifted)")
plt.legend()

In [ ]:
plt.scatter(df["r"] - df["i"] + d_ri, df["r"] + d_r, s=0.5, lw=0, zorder=-5)
plt.ylim(27, 16)
plt.xlim(-1.2, 1.8)
filter_utils.plot_iso_ri(params,)


plt.plot(julen_ri, julen_r, label="paper isochrone", color="C1")

plt.plot([], [], label="r-i isochrone", color="grey")

plt.gca().set_box_aspect(1)
plt.xlabel("$r - i$ (mag)")
plt.ylabel("$r$ (mag)")
plt.title("NGC 5466 (shifted)")
# plt.legend()

In [ ]:
plt.scatter(df["r"] - df["i"], df["r"], s=0.5, lw=0, zorder=-5)
plt.ylim(26.5, 16)
plt.xlim(-1, 2)

plt.gca().set_box_aspect(1)

filter_utils.plot_iso_ri(params)
plt.xlabel("$r - i$ (mag)")
plt.ylabel("$r$ (mag)")
plt.title("NGC 5466")

In [ ]:
plt.scatter(df["ra"], df["dec"], s=1)


# Varying the isochrone parameters

In [ ]:
from copy import copy

In [ ]:
import arya

In [ ]:
def plot_ri_iso(dm=dm, A_V=0.1, iso_fe_h="m2.00", iso_log_age=10.1, **kwargs):
    iso = isochrones[iso_fe_h][iso_log_age]
    iso = iso[iso["phase"] < 4]

    A_g, A_r, A_i = get_extinction(A_V)
    plt.plot(iso["SDSS_r"] - iso["SDSS_i"] + A_r - A_i, iso["SDSS_r"] + dm + A_r, zorder=-5, **kwargs)

    plt.xlabel("r - i")
    plt.ylabel("r")
    plt.gca().invert_yaxis()

In [ ]:
def plot_gr_iso(dm=dm, A_V=0.1, iso_fe_h="m2.00", iso_log_age=10.1, **kwargs):
    iso = isochrones[iso_fe_h][iso_log_age]
    iso = iso[iso["phase"] < 4]

    A_g, A_r, A_i = get_extinction(A_V)
    plt.plot(iso["SDSS_g"] - iso["SDSS_r"] + A_g - A_r, iso["SDSS_g"] + dm + A_g, zorder=-5, **kwargs)

    plt.xlabel("g - r")
    plt.ylabel("g")
    plt.gca().invert_yaxis()

In [ ]:
hm = arya.HueMap((0, 0.5))

fig, axs = plt.subplots(1, 2)

plt.sca(axs[0])
for A_V in [0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    plot_gr_iso(A_V=A_V, color=hm(A_V), label="A_V = " + str(A_V))

filter_utils.plot_iso_gr(params, gr_err)
plt.xlim(0, 1.5)
plt.ylim(25, 12)


plt.sca(axs[1])
for A_V in [0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    plot_ri_iso(A_V=A_V, color=hm(A_V), label="A_V = " + str(A_V))
plt.ylim(25, 12)
filter_utils.plot_iso_ri(params, ri_err)
plt.xlim(-0.5, 1)


arya.Legend(-1)
plt.ylim(25, 12)

In [ ]:
hm = arya.HueMap((1e9, 2e10))

fig, axs = plt.subplots(1, 2)
plt.sca(axs[0])

for age in [9.5, 9.7, 9.9, 10, 10.1, 10.2, 10.3]:
    plot_gr_iso(iso_log_age=age, color=hm(10**age), label=str(round(10**age / 1e9)) + " Gyr")
plt.ylim(25, 12)
filter_utils.plot_iso_gr(params, gr_err)
plt.xlim(0, 1.5)


plt.sca(axs[1])
for age in [9.5, 9.7, 9.9, 10, 10.1, 10.2, 10.3]:
    plot_ri_iso(iso_log_age=age, color=hm(10**age), label=str(round(10**age / 1e9)) + " Gyr")
filter_utils.plot_iso_ri(params, ri_err)
plt.xlim(-0.5, 1)

arya.Legend(-1)
plt.ylim(25, 12)

In [ ]:
hm = arya.HueMap((-3, -1))
fig, axs = plt.subplots(1, 2)
plt.sca(axs[0])


for fe_h in isos_fe_h:
    x = -float(fe_h[1:])
    print(x)
    plot_gr_iso(iso_fe_h=fe_h, color=hm(x), label=fe_h)
plt.ylim(25, 12)
filter_utils.plot_iso_gr(params, gr_err)
plt.xlim(0, 1.5)


plt.sca(axs[1])

for fe_h in isos_fe_h:
    x = -float(fe_h[1:])
    print(x)
    plot_ri_iso(iso_fe_h=fe_h, color=hm(x), label=fe_h)
filter_utils.plot_iso_ri(params, ri_err)
plt.xlim(-0.5, 1)

arya.Legend(-1)
plt.ylim(25, 12)